In [1]:
%load_ext autoreload
%autoreload 2

In [12]:
import random
import numpy as np
import torch
import pandas as pd

from pathlib import Path
from flowrec.data.datamodule import AmazonReviewDataConfig, AmazonReviewDataModule
from flowrec.models.sasrec import SASRecConfig, SASRec
from flowrec.ft_methods.sft import SFTConfig, SFTMethod
from flowrec.training import TrainerConfig, Trainer

from flowrec.ft_methods.sft.metrics import item_ranking_metrics

In [3]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [4]:
@torch.no_grad()
def compute_test_hit_ndcg_coverage(
    model: torch.nn.Module,
    dm: AmazonReviewDataModule,
    *,
    ks=(5, 10, 20),
    pad_item_id: int = 0,
) -> dict[str, float]:
    """Compute NDCG@K, HitRate@K, Coverage@K on test data
    """
    device = next(model.parameters()).device
    model.eval()

    vocab_size = dm.item_vocab_size
    recommendable = max(vocab_size - 1, 1)

    # Accumulate instance-level hit and ndcg
    hits = {k: [] for k in ks}
    ndcg_sums = {k: 0.0 for k in ks}
    ndcg_counts = {k: 0 for k in ks}

    recommended_sets = {k: set() for k in ks}

    for batch in dm.test_dataloader():
        for key, val in batch.items():
            if torch.is_tensor(val):
                batch[key] = val.to(device, non_blocking=True)

        out = model(batch)
        scores = out.scores  # [B, V]
        target_item_ids = batch["target_item_ids"]

        targets = target_item_ids.long()
        valid = targets != pad_item_id
        if valid.sum() == 0:
            continue

        max_k = max(ks)
        topk_items = torch.topk(scores, k=max_k, dim=-1).indices  # [B, max_k]
        topk_valid = topk_items[valid]
        targets_valid = targets[valid]

        for k in ks:
            kk = int(k)
            topk_k = topk_valid[:, :kk]

            matches = topk_k.eq(targets_valid.unsqueeze(1))
            hit = matches.any(dim=1)
            hits[k].append(hit.float().cpu())

            rank_pos = matches.float().argmax(dim=1)
            ndcg = torch.zeros_like(hit, dtype=torch.float32)
            if hit.any():
                ndcg[hit] = 1.0 / torch.log2(rank_pos[hit].float() + 2.0)
            ndcg_sums[k] += float(ndcg.mean().item())
            ndcg_counts[k] += 1

            uniq = torch.unique(topk_k).detach().cpu().tolist()
            recommended_sets[k].update(int(x) for x in uniq if int(x) != pad_item_id)

    metrics: dict[str, float] = {}
    for k in ks:
        hit_t = torch.cat(hits[k], dim=0) if hits[k] else torch.zeros(0)
        hit_rate = float(hit_t.mean().item()) if hit_t.numel() else 0.0
        coverage = len(recommended_sets[k]) / recommendable
        ndcg = ndcg_sums[k] / max(ndcg_counts[k], 1)
        metrics[f"ndcg@{k}"] = ndcg
        metrics[f"hit_rate@{k}"] = hit_rate
        metrics[f"coverage@{k}"] = coverage

    return metrics

In [5]:
reviews_path = Path("../data/raw/reviews_Beauty.json.gz")
metadata_path = Path("../data/raw/meta_Beauty.json.gz")

data_cfg = AmazonReviewDataConfig(
    reviews_path=reviews_path,
    metadata_path=metadata_path,
    semantic_id_path=None,
    min_user_interactions=5,
    min_item_interactions=5,
    split_strategy="leave_one_out",
    max_history_len=50,
    history_padding="right",
    batch_size=256,
    eval_batch_size=512,
)

data_cfg = AmazonReviewDataConfig(
    reviews_path=reviews_path,
    metadata_path=metadata_path,
    item_key="parent_asin",

    # Global time split.
    split_strategy="global_time",
    global_val_ratio=0.1,
    global_test_ratio=0.1,
    global_history_policy="rolling",
    global_eval_target_mode="last_per_user",

    # Requested config.
    train_all_prefixes=False,
    min_user_interactions=1,
    min_item_interactions=1,
    max_history_len=20,

    # Batching.
    batch_size=256,
    eval_batch_size=256,
    num_workers=0,
    seed=42,
)

dm = AmazonReviewDataModule(data_cfg)
dm.prepare_data()
dm.setup()

print("Dataset summary:")
print(dm.summary())
print("Item vocab size:", dm.item_vocab_size)

Dataset summary:
{'num_users': 322870, 'num_items': 199005, 'num_train_examples': 254165, 'num_val_examples': 54711, 'num_test_examples': 55239, 'num_interactions': 2023070, 'split_strategy': 'global_time', 'global_val_cutoff': 1394323200, 'global_test_cutoff': 1400284800, 'sid_len': None, 'item_vocab_size': 199006}
Item vocab size: 199006


In [6]:
model_cfg = SASRecConfig(
    item_vocab_size=dm.item_vocab_size,
    max_history_len=data_cfg.max_history_len,
    d_model=256,
    nhead=2,
    num_layers=2,
    dropout=0.3,
    pos_drop=0.2
)

sft_cfg = SFTConfig()
method_cfg = sft_cfg

trainer_cfg = TrainerConfig(
    max_epochs=10,
    gradient_accumulation_steps=4,
    use_amp=True,
    log_every_steps=50,
    validate_every_epochs=1,
    monitor="val/loss",
    early_stopping_patience=3,
    checkpoint_dir="../artifacts/sasrec_beauty",
)

In [7]:
seeds = [42, 43, 44]
ks = (5, 10, 20)
all_seed_metrics = []

for seed in seeds:
    print(f"\n===== Train/eval seed={seed} =====")
    set_seed(seed)

    model = SASRec(model_cfg)
    method = SFTMethod(method_cfg)

    trainer = Trainer.from_optimizer_config(
        model=model,
        method=method,
        cfg=trainer_cfg,
        lr=1e-3,
        weight_decay=1e-2,
    )

    _ = trainer.fit(dm.train_dataloader(), dm.val_dataloader())

    ckpt_path = f"../artifacts/sasrec_beauty/final_sasrec_seed_{seed}.pt"
    model.save_checkpoint(ckpt_path)

    test_metrics = compute_test_hit_ndcg_coverage(model, dm, ks=ks)
    test_metrics["seed"] = seed

    print("Test metrics:")
    for key in sorted([k for k in test_metrics.keys() if k != 'seed']):
        print(f"  {key}: {test_metrics[key]:.6f}")

    all_seed_metrics.append(test_metrics)


===== Train/eval seed=42 =====


/Users/maksimshuklin/hse/flowrec/flowrec/training/trainer.py:183: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self._scaler = torch.cuda.amp.GradScaler(enabled=self._use_amp)


Test metrics:
  coverage@10: 0.340795
  coverage@20: 0.391196
  coverage@5: 0.279395
  hit_rate@10: 0.029870
  hit_rate@20: 0.038922
  hit_rate@5: 0.022412
  ndcg@10: 0.016533
  ndcg@20: 0.018817
  ndcg@5: 0.014122

===== Train/eval seed=43 =====
Test metrics:
  coverage@10: 0.333198
  coverage@20: 0.381523
  coverage@5: 0.275315
  hit_rate@10: 0.030033
  hit_rate@20: 0.038723
  hit_rate@5: 0.022665
  ndcg@10: 0.016324
  ndcg@20: 0.018512
  ndcg@5: 0.013942

===== Train/eval seed=44 =====
Test metrics:
  coverage@10: 0.337162
  coverage@20: 0.386724
  coverage@5: 0.277928
  hit_rate@10: 0.030594
  hit_rate@20: 0.038415
  hit_rate@5: 0.023154
  ndcg@10: 0.016703
  ndcg@20: 0.018677
  ndcg@5: 0.014303


In [13]:
df = pd.DataFrame(all_seed_metrics)

metric_cols = sorted([c for c in df.columns if c != "seed"])
df = df[["seed"] + metric_cols]
df = df.round(6)

df

,seed,coverage@10,coverage@20,coverage@5,hit_rate@10,hit_rate@20,hit_rate@5,ndcg@10,ndcg@20,ndcg@5
0,42,0.340795,0.391196,0.279395,0.029870,0.038922,0.022412,0.016533,0.018817,0.014122
1,43,0.333198,0.381523,0.275315,0.030033,0.038723,0.022665,0.016324,0.018512,0.013942
2,44,0.337162,0.386724,0.277928,0.030594,0.038415,0.023154,0.016703,0.018677,0.014303


In [14]:
df.to_csv("metrics_10_epochs.csv", index=False)

In [16]:
df_mean_std = pd.DataFrame({
    col: [f"{df[col].mean():.3f} ± {3 * df[col].std():.3f}"]
    for col in df.select_dtypes(include='number').columns
})
df_mean_std = df_mean_std.drop(columns=['seed'])
df_mean_std

,coverage@10,coverage@20,coverage@5,hit_rate@10,hit_rate@20,hit_rate@5,ndcg@10,ndcg@20,ndcg@5
0,0.337 ± 0.011,0.386 ± 0.015,0.278 ± 0.006,0.030 ± 0.001,0.039 ± 0.001,0.023 ± 0.001,0.017 ± 0.001,0.019 ± 0.000,0.014 ± 0.001


In [17]:
import pandas as pd

data = {
    'Model': ['SASRec', 'TIGER'],
    'Dataset': ['Beauty', 'Beauty'],
    'N@5': [0.02087, 0.02524],
    'N@10': [0.02718, 0.03191],
    'N@20': [0.03447, 0.03940],
    'H@5': [0.03197, 0.03756],
    'H@10': [0.0516, 0.0582]
}

df = pd.DataFrame(data)

df

,Model,Dataset,N@5,N@10,N@20,H@5,H@10
0,SASRec,Beauty,0.02087,0.02718,0.03447,0.03197,0.0516
1,TIGER,Beauty,0.02524,0.03191,0.03940,0.03756,0.0582


In [18]:
df.to_csv("data_new.csv", index=False)

In [19]:
trainer.state

TrainerState(epoch=3, global_step=3972, optimizer_step=996, best_metric=11.667084612034694, best_epoch=0, bad_epochs=3, should_stop=True, history=[{'epoch': 0, 'global_step': 993, 'optimizer_step': 249, 'train': {'train/loss': 11.49493849101472, 'train/sft/base_loss': 11.49493849101472, 'train/sft/loss': 11.49493849101472, 'train/sft/ndcg@10': 0.005524159109835978, 'train/sft/ndcg@20': 0.006909643135796462, 'train/sft/ndcg@5': 0.0043936602959477765, 'train/sft/recall@10': 0.010170558496139764, 'train/sft/recall@20': 0.015694529144013696, 'train/sft/recall@5': 0.006653158382040656}, 'val': {'val/loss': 11.667084612034694, 'val/sft/base_loss': 11.667084612034694, 'val/sft/loss': 11.667084612034694, 'val/sft/ndcg@10': 0.004481032806397077, 'val/sft/ndcg@20': 0.005699219767199875, 'val/sft/ndcg@5': 0.003549211284874654, 'val/sft/recall@10': 0.009212041454186543, 'val/sft/recall@20': 0.014000840781561295, 'val/sft/recall@5': 0.006269305989654732}, 'best_checkpoint': '../artifacts/sasrec_bea

In [20]:
with open('trainer_state.txt', 'w') as f:
    print(str(trainer.state), file=f)